# 05 — Checkpoint Analysis

Use this notebook to:
1. List all variables in the **old** checkpoint (`initial_checkpoint_folder/ckpt-920304`)
2. List all variables in the **new** model (after building it)
3. Compare the two lists — find matches and gaps
4. Inform the variable name mapping in `tools/checkpoint_migration.py`

Run this **before** attempting the migration script.

In [ ]:
import sys
sys.path.insert(0, '..')  # repo root

import tensorflow as tf
print('TF version:', tf.__version__)

## 1. List old checkpoint variables

In [ ]:
OLD_CKPT = '../initial_checkpoint_folder/ckpt-920304'

from tools.checkpoint_migration import list_checkpoint_variables
old_vars = list_checkpoint_variables(OLD_CKPT)
print(f'Old checkpoint has {len(old_vars)} variables')

In [ ]:
import pandas as pd

rows = [(name, shape, dtype) for name, (shape, dtype) in sorted(old_vars.items())]
df_old = pd.DataFrame(rows, columns=['name', 'shape', 'dtype'])
pd.set_option('display.max_colwidth', 120)
df_old

In [ ]:
# Which top-level namespaces exist in the old checkpoint?
namespaces = sorted(set(n.split('/')[0] for n in old_vars))
print('Top-level namespaces:', namespaces)

## 2. Build new model and list its variables

In [ ]:
from configs.yaml_loader import load_config
from models.yolo_v8 import YoloV8

cfg = load_config('../configs/experiments/yolo/yolov8_poly_dist.yaml')
model = YoloV8(cfg.task.model)

# Build model by doing a dummy forward pass
dummy = tf.zeros([1] + cfg.task.model.input_size)
_ = model(dummy, training=False)

new_vars = {v.name.rstrip(':0'): (tuple(v.shape), v.dtype.name) for v in model.variables}
print(f'New model has {len(new_vars)} variables')

In [ ]:
rows_new = [(name, shape, dtype) for name, (shape, dtype) in sorted(new_vars.items())]
df_new = pd.DataFrame(rows_new, columns=['name', 'shape', 'dtype'])
df_new

## 3. Dry-run the name mapping

In [ ]:
from tools.checkpoint_migration import build_name_mapping

mapping = build_name_mapping(
    old_vars=old_vars,
    new_vars=new_vars,
    modules=['backbone', 'decoder'],
)

print(f'\nMatched: {len(mapping)} variable pairs')
for old, new in sorted(mapping.items()):
    old_shape = old_vars[old][0]
    print(f'  {old:<80} →  {new}  {old_shape}')

## 4. Identify unmatched old variables (from backbone/decoder)

These need manual inspection.  If they are batch-norm statistics or optimizer
slots they can safely be ignored.

In [ ]:
modules = ['backbone', 'decoder']
old_backbone_vars = {k: v for k, v in old_vars.items() if any(m in k for m in modules)}
matched_old = set(mapping.keys())
unmatched = {k: v for k, v in old_backbone_vars.items() if k not in matched_old}

print(f'Unmatched backbone/decoder variables: {len(unmatched)}')
for name, (shape, dtype) in sorted(unmatched.items()):
    print(f'  {name:<80} {shape}')

## 5. Check total parameter counts

Sanity check: backbone + decoder should have similar param counts between old and new.

In [ ]:
import numpy as np

def count_params(var_dict, modules):
    total = 0
    for name, (shape, _) in var_dict.items():
        if any(m in name for m in modules):
            total += int(np.prod(shape))
    return total

modules = ['backbone', 'decoder']
old_count = count_params(old_vars, modules)
new_count = count_params(new_vars, modules)

print(f'Old backbone+decoder params: {old_count:,}')
print(f'New backbone+decoder params: {new_count:,}')
print(f'Ratio: {new_count / old_count:.3f}  (should be ~1.0 if architectures match)')